# ప్రాధాన్యత సభ్య మిడ్‌లేయర్‌తో హోటల్ బుకింగు

ఈ నోట్‌బుక్ Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్‌ను ఉపయోగించి **ఫంక్షన్-ఆధారిత మిడ్‌లేయర్**ని చూపిస్తుంది. మేము షరతుపరపు వర్క్‌ఫ్లో ఉదాహరణపై నిర్మించుకుంటూ, ప్రాధాన్యత సభ్యులకు ప్రత్యేక హక్కులు ఇచ్చే మిడ్‌లేయర్ పొరని కలిపి చేస్తాము.

## మీరు నేర్చుకునేది:
1. **ఫంక్షన్-ఆధారిత మిడ్‌లేయర్**: ఫంక్షన్ ఫలితాలను అంతరించు మరియు మార్చు
2. **కాంటెక్స్ట్ యాక్సెస్**: ఎగ్జిక్యూషన్ తర్వాత `context.result` ను చదవడం మరియు మార్చడం
3. **వ్యవసాయ లాజిక్ అమలు**: ప్రాధాన్యత సభ్య ప్రయోజనాలు
4. **ఫలిత వేరియేషన్**: వినియోగదారుని స్థితి ఆధారంగా ఫంక్షన్ ఫలితాలను మార్చడం
5. **ఆదే వర్క్‌ఫ్లో, విభిన్న ఫలితాలు**: మిడ్‌లేయర్ ఆధారిత ప్రవర్తనా మార్పులు

## మిడ్‌లేయర్‌తో వర్క్‌ఫ్లో నిర్మాణం:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## షరతుపర ఊర్థై నుండి ప్రధాన తేడా:

**మిడ్‌లేయర్ లేకుండా** (14-conditional-workflow.ipynb):
- పారిస్ వద్ద గది లేనిప్పుడు → alternative_agent కి మార్గనిర్దేశం

**మిడ్‌లేయర్‌తో** (ఈ నోట్‌బుక్):
- సాధారణ వినియోగదారు + పారిస్ → గది లేని పరిస్థితి → alternative_agent కి మార్గనిర్దేశం
- ప్రాధాన్యత వినియోగదారు + పారిస్ → 🌟 మిడ్‌లేయర్ అంతరించు! → గది అందుబాటులో → booking_agent కి మార్గనిర్దేశం

## అవసరమైన విషయాలు:
- Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ ఇన్‌స్టాల్ చేయబడింది
- షరతుపర workflows గురించి అవగాహన (14-conditional-workflow.ipynb చూడండి)
- GitHub టోకన్ లేదా OpenAI API కీ
- మిడ్‌లేయర్ నమూనాలు గురించి ప్రాథమిక అవగాహన


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## దశ 1: నిర్మిత అవుట్‌పుట్‌ల కోసం Pydantic మోడల్స్ నిర్వచించండి

ఈ మోడల్స్ ఏజెంట్లు తిరిగి ఇచ్చే **స్కీమాను** నిర్వచిస్తాయి. మిడిల్వేర్ అందుబాటులో ఉన్న ఫలితాన్ని సవరించినప్పుడు ట్రాక్ చేయడానికి మేము `priority_override` ఫీల్డ్‌ని జోడించాము.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## దశ 2: ప్రాధాన్యత సభ్యుల డేటాబేస్ నిర్వచించండి

ఈ డెమో కోసం, మనం ప్రాధాన్యత సభ్యత్వ డేటాబేస్‌ను సిమ్యులేట్ చేస్తాము. ఉత్పత్తిలో, ఇది ఒక వాస్తవ డేటాబేస్ లేదా API ని క్వారీ చేస్తుంది.

**ప్రాధాన్యత సభ్యులు:**
- `alice@example.com` - VIP సభ్యుడు
- `bob@example.com` - ప్రీమియం సభ్యుడు  
- `priority_user` - టెస్ట్ ఖాతా


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## దశ 3: హోటల్ బుకింగ్ టూల్ సృష్టించండి

షరతుపూర్వక వర్క్‌ఫ్లో లాంటిది, కానీ ఇప్పుడు ఇది మిడిల్‌వేర్ ద్వారా అడ్డుకుపోతుంది!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## దశ 4: 🌟 ప్రాధాన్యత తనిఖీ మిడిల్వేర్ సృష్టించడం (ముఖ్య లక్షణం!)

ఇది ఈ నోట్బుక్ యొక్క **ప్రధాన కార్యాచరణ**. మిడిల్వేర్:

1. **హోటల్_బుకింగ్** ఫంక్షన్ కాల్‌ను అడ్డుకుంటుంది
2. `next(context)`ను పిలిచి సాధారణంగా ఫంక్షన్‌ను **నిష్పాదన చేస్తుంది**
3. `context.result`లో ఫలితాన్ని **పరిశీలిస్తుంది**
4. యూజర్ ప్రాధాన్యత ఉన్నప్పుడూ మరియు గది లభించకపోతే ఫలితాన్ని **మారుస్తుంది**
5. మినహాయించిన ఫలితాన్ని ఏజెంట్‌కు **తిరిగివస్తుంది**

**ప్రధాన నమూనా:**
```python
async def my_middleware(context, next):
    await next(context)  # ఫంక్షన్‌ను అమలు చేయండి
    # ఇప్పుడు context.result లో ఫంక్షన్ ఫలితం ఉంటుంది
    if some_condition:
        context.result = new_value  # మించినది!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## దశ 5: రూటింగ్ కోసం పరిస్థితి ఫంక్షన్లు నిర్వచించండి

సాంప్రదాయక వర్క్‌ఫ్లోకు సమానమైన పరిస్థితి ఫంక్షన్లు - అవి రూటింగ్‌ను నిర్ణయించడానికి నిర్మిత అవుట్‌పుట్‌ను పరిశీలిస్తాయి.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## దశ 6: కస్టమ్ డిస్ప్లే ఎగ్జిక్యూటర్ సృష్టించండి

మునుపటి లాంటిదే ఎగ్జిక్యూటర్ - తుది వర్క్‌ఫ్లో అవుట్పుట్ ను ప్రదర్శిస్తుంది.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## దశ 7: ముట్టడి వాతావరణ మార్పిడులను లోడ్ చేయండి

LLM క్లయింట్ (Microsoft Foundry / Azure OpenAI) ను constituency చేయండి.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## దశ 8: మీడిల్వేర్‌తో AI ఏజెంట్లను సృష్టించడం

**ముఖ్య తేడా:** availability_agent సృష్టిస్తున్నప్పుడు, మేము `middleware` పరామితిని పంపిణీ చేస్తాము!

ఇలా priority_check_middleware ను ఏజెంట్ యొక్క ఫంక్షన్ ఆహ్వానం పైప్‌లైన్‌లో ఇంజెక్ట్ చేస్తాము.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## దశ 9: వర్క్‌ఫ్లోని నిర్మించండి

ముందటి వంటిది అదే వర్క్‌ఫ్లో నిర్మాణం - అందుబాటుపై ఆధారపడి షరతుగల రూటింగ్.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## దశ 10: టెస్ట్ కేస్ 1 - ప్యారిస్‌లో సాధారణ వినియోగదారు (ఒవర్‌రైడ్ లేదు)

ఒక సాధారణ వినియోగదారు ప్యారిస్ బుక్ చేయడానికి ప్రయత్నిస్తాడు → గదులు లేవు → alternative_agent కి మార్గదర్శనం చేస్తుంది


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## దశ 11: పరీక్ష కేసు 2 - 🌟 ప్రాధాన్యత కలిగొన్న వినియోగదారు పారిస్ లో (ఓవర్‌రైడ్‌తో!)

ఒక ప్రాధాన్యత సభ్యుడు పారిస్ బుక్ చేయడానికి ప్రయత్నిస్తాడు → మొదటిసారిగా గదులు లేరు → 🌟 మిడిల్‌వేర్ ఓవర్‌రైడ్ చేస్తుంది! → booking_agent కి రూట్ చేయబడుతుంది

**ఇది మిడిల్‌వేర్ శక్తి యొక్క ప్రధాన ప్రదర్శన!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## దశ 12: పరీక్ష కేసు 3 - స్టాక్‌హోలం లో ప్రాధాన్యత ఉన్న వినియోగదారు (ఇప్పటికే అందుబాటులో ఉంది)

ప్రాధాన్యత వినియోగదారు స్టాక్‌హోలం ప్రయత్నిస్తారు → గదులు అందుబాటులో ఉన్నాయి → ఓవర్‌రైడ్ అవసరం లేదు → booking_agent కు మార్గం ఏర్పడుతుంది

ఇది చూపిస్తుంది మిడిల్వేర్ అవసరమైనప్పుడు మాత్రమే పనిచేస్తుందని!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ముఖ్యమైన విషయాలు మరియు మిడిల్వేర్ కాన్సెప్ట్‌లు

### ✅ మీరు నేర్చుకున్నవి:

#### **1. ఫంక్షన్ ఆధారిత మిడిల్వేర్ నమూనా**

మిడిల్వేర్ సింపుల్ అసింక్రోనస్ ఫంక్షన్ ఉపయోగించి ఫంక్షన్ కాల్స్‌ను ఇంటర్‌సెప్ట్ చేస్తుంది:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ఫంక్షన్ అమలుకు ముందు
    print("Intercepting...")
    
    # ఫంక్షన్ ను అమలు చేయండి
    await next(context)
    
    # ఫంక్షన్ అమలుకు తర్వాత - ఫలితాన్ని పరిశీలించండి
    if context.result:
        # అవసరమైతే ఫలితాన్ని మార్పు చేయండి
        context.result = modified_value
```

#### **2. కాంటెక్స్ట్ యాక్సెస్ మరియు ఫలితాన్ని మార్చడం**

- `context.function` - పిలవబడుతోన్న ఫంక్షన్‌ను ప్రాప్తి చేసుకోండి
- `context.arguments` - ఫంక్షన్ ఆర్గ్యుమెంట్లను చదవండి
- `context.kwargs` - అదనపు పారామీటర్లను యాక్సెస్ చేయండి
- `await next(context)` - ఫంక్షన్‌ను అమలు చేయండి
- `context.result` - ఫంక్షన్ అవుట్‌పుట్‌ను చదవండి/మార్చండి

#### **3. బిజినెస్ లాజిక్ అమలుచేయడం**

మా మిడిల్వేర్ ప్రాధాన్యత సభ్యుల లాభాలను అమలు చేస్తుంది:
- **ప్రామాణిక సభ్యులు**: ఎలాంటి మార్పులు లేవు, సాంప్రదాయ వర్క్‌ఫ్లో
- **ప్రాధాన్యత సభ్యులు**: "అభావం లేదు"ని → "ఉందాలి"గా మార్చడం
- **నిబంధనాత్మక లాజిక్**: అవసరమైతే మాత్రమే మార్చడం

#### **4. అదే వర్క్‌ఫ్లో, వేరే ఫలితాలు**

మిడిల్వేర్ శక్తి:
- ✅ వర్క్‌ఫ్లో స్థాపనలో మార్పులు లేవు
- ✅ టూల్ ఫంక్షన్‌లో మార్పులు లేవు
- ✅ నిబంధనాత్మక రూటింగ్ లాజిక్‌లో మార్పులు లేవు
- ✅ కేవలం మిడిల్వేర్ → వేరే ప్రవర్తన!

### 🚀 వాస్తవ ప్రపంచంలో అనువర్తనలు:

1. **VIP/ప్రీమియం ఫీచర్లు**
   - ప్రీమియం వినియోగదారుల కోసం రేట్ పరిమితులను మార్చడం
   - వనరులకు ప్రాధాన్యత యాక్సెస్ ఇవ్వడం
   - ప్రీమియం ఫీచర్లను డైనమిక్‌గా అందించడం

2. **A/B టెస్టింగ్**
   - వినియోగదారులను వేరే అమలులకు రూట్ చేయడం
   - ప్రత్యేక వినియోగదారులతో కొత్త ఫీచర్లను పరీక్షించడం
   - దశ పద్ధతిలో ఫీచర్ల విడుదల

3. **భద్రత మరియు అనుగుణత**
   - ఫంక్షన్ కాల్స్ ఆడిట్ చేయడం
   - సున్నితమైన చర్యలను బ్లాక్ చేయడం
   - బిజినెస్ నియమాలను అమలు చేయడం

4. **పర్ఫార్మెన్స్ ఆప్టిమైజేషన్**
   - నిర్దిష్ట వినియోగదారుల కోసం ఫలితాలను క్యాష్ చేయడం
   - సాధ్యమైనప్పుడు ఖరీదు ఎక్కువ చేసే చర్యలను తప్పించడం
   - డైనమిక్ వనరుల కేటాయింపు

5. **లోప నిహారిక & రీట్రై**
   - లోపాలను సౌమ్యంగా హ్యాండిల్ చేయడం
   - రీట్రై లాజిక్‌ను అమలు చేయడం
   - ప్రత్యామ్నాయ అమలులకుFallback

6. **లాగింగ్ & మానిటరింగ్**
   - ఫంక్షన్ అమలయిన సమయాలు ట్రాక్ చేయడం
   - పారామీటర్లు మరియు ఫలితాలను లాగ్ చేయడం
   - వినియోగ నమూనాలను పరిశీలించడం

### 🔑 డెకరేటర్ల నుండి కీలక తేడాలు:

| ఫీచర్ | డెకరేటర్ | మిడిల్వేర్ |
|---------|-----------|------------|
| **వ్యాప్తి** | ఒకే ఫంక్షన్ | ఏజెంట్‌లో అన్ని ఫంక్షన్లు |
| **అడపాదడపా మార్పు** | నిర్వచన సమయంలో స్థిరంగా | రన్‌టైమ్‌లో డైనమిక్ |
| **కాంటెక్స్ట్** | పరిమిత | పూర్తిగా ఏజెంట్ కాంటెక్స్ట్ |
| **సంయోజనం** | బహుళ డెకరేటర్లు | మిడిల్వేర్ పైప్‌లైన్ |
| **ఏజెంట్ అగివేర్** | కాదు | అవును (ఏజెంట్ స్థితిని యాక్సెస్) |

### 📚 언제 మిడిల్వేర్ ఉపయోగించాలి:

✅ **మిడిల్వేర్ ఉపయోగించండి:**
- మీరు యూజర్/సెషన్ స్థితిపై ఆధారపడి ప్రవర్తనను మార్చాలి
- బహుళ ఫంక్షన్లపై లాజిక్ వర్తించాలి
- ఏజెంట్ స్థాయి కాంటెక్స్ట్ యాక్సెస్ అవసరం
- మీరు క్రాస్-కట్టింగ్ అంశాలు (లాగింగ్, ఆథ్, మొదలైనవి) అమలుచేస్తున్నార

❌ **మిడిల్వేర్ ఉపయోగించకండి:**
- సింపుల్ ఇన్‌పుట్ వాలిడేషన్ (Pydantic ఉపయోగించండి)
- ఫంక్షన్-స్పెసిఫిక్ లాజిక్ (ఫంక్షన్‌లో ఉంచండి)
- ఒకసారి మార్పులు (కేవలం ఫంక్షన్‌ను మార్చండి)

### 🎓 అధునాతన నమూనాలు:

```python
# బహుళ మిడిల్‌వేర్ (నిర్వహణ క్రమం ముఖ్యం!)
middleware=[
    logging_middleware,      # మొదట లాగ్స్
    auth_middleware,         # ఆపై ఆథ్ తనిఖీ
    cache_middleware,        # ఆపై క్యాష్ తనిఖీ
    rate_limit_middleware,   # ఆపై రేటు పరిమితులు
    priority_check_middleware  # చివరికి ప్రాధాన్యత తనిఖీ
]

# షరతు ఆధారిత మిడిల్‌వేర్ నిర్వాహణ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ఫలితాన్ని మార్చండి
    else:
        # పూర్తి నిర్వాహణను స్కిప్ చేయండి
        context.result = cached_value
```

### 🔗 సంబంధిత కాన్సెప్ట్‌లు:

- **ఏజెంట్ మిడిల్వేర్**: agent.run() కాల్‌లను ఇంటర్‌సెప్ట్స్ చేస్తుంది
- **ఫంక్షన్ మిడిల్వేర్**: టూల్ ఫంక్షన్ కాల్స్‌ను ఇంటర్‌సెప్ట్స్ చేస్తుంది (మేము ఉపయోగించినది!)
- **మిడిల్వేర్ పైప్‌లైన్**: అనుక్రమంలో అమలు అయ్యే మిడిల్వేర్ శ్రేణి
- **కాంటెక్స్ట్ ప్రొపగేషన్**: మిడిల్వేర్ చైన్ ద్వారా స్థితిని పంపడం


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
